# SparkClient: Connecting to an Existing Spark Connect Session

This notebook demonstrates the two-client ("bring your own server") pattern:
1. **Setup Client**: Provisions the Spark Connect server on Kubernetes and resolves its service URL.
2. **Consumer Client**: Attaches to the running remote server using `client.connect(base_url=...)` without provisioning new cluster resources.
3. **Execution & Cleanup**: Executes distributed transformations and deletes the server resource.

## 1. Imports and Backend Setup

Configure backend client settings and import connection helpers.

In [ ]:
import os
import uuid

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import Name, SparkClient
from kubeflow.spark.backends.kubernetes.utils import build_service_url

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "spark-test")
backend_config = KubernetesBackendConfig(namespace=namespace)
print(f"Backend configured for namespace: {namespace}")

## 2. Phase 1: Provision Spark Connect Server (Setup Client)

Create the Spark Connect server instance and extract its connection URL (`sc://...`).

In [ ]:
session_name = f"connect-existing-{uuid.uuid4().hex[:8]}"
setup_client = SparkClient(backend_config=backend_config)

print(f"Deploying Spark Connect server: {session_name}...")
setup_spark = setup_client.connect(options=[Name(session_name)], timeout=180)

info = setup_client.get_session(session_name)
service_url = build_service_url(info)
print("Server deployed successfully!")
print(f"Session Name: {session_name}")
print(f"Service URL:  {service_url}")

# Disconnect local driver while keeping the server running on Kubernetes
setup_spark.stop()
print("Setup client disconnected. Server remains running on cluster.")

## 3. Phase 2: Connect via base_url (Consumer Client)

Initialize an independent client and connect to the existing server using its direct service URL.

In [ ]:
print(f"Connecting consumer client to {service_url}...")
test_client = SparkClient(backend_config=backend_config)
test_spark = test_client.connect(base_url=service_url)
print("Consumer client connected successfully!")

## 4. Phase 3: Execute Spark Operations

Run queries against the existing Spark cluster.

In [ ]:
count = test_spark.range(100).count()
print(f"spark.range(100).count() = {count}")
assert count == 100, f"Expected 100, got {count}"

test_spark.stop()
print("Consumer session disconnected.")

## 5. Phase 4: Clean Up Server Resources

Tear down the Spark Connect Kubernetes deployment.

In [ ]:
print(f"Deleting Spark Connect server: {session_name}")
setup_client.delete_session(session_name)
print("Server resource deleted successfully.")